In [65]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [66]:
from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers


import os
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime, date, timedelta


# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.ingester.IngesterClass import Ingester
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta
class SurveySummaryIngester(Ingester):
    def __init__(self, arguments):
        super().__init__(arguments)
        self.table = KPI_SurveySummary

    def update_check(self):
        customer_name = self.customer_info['Name']
        customer_id = self.customer_info['CustomerId']
        customer_db = self.customer_info['DBLocation']

        self.Logger.info(f"Processing customer: {customer_name}")
        self.Logger.info(f"Getting reports from {self.update_window} to {self.current_date}")
        #Query the last report
        self.data['reports'] = Query(
            f"""
            SELECT ReportId, ReportDate, LastUpdated FROM KPI_ReportSummary
            WHERE CustomerId = '{self.customer_info['CustomerId']}'
            ORDER BY LastUpdated DESC
            """
        ).execute(KPIHub_Conn)

        self.data['survey_count'] = Query(
            f"""
            SELECT COUNT(*) as SurveyCount
            FROM KPI_SurveySummary
            WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{customer_id}')
            """
        ).execute(KPIHub_Conn)

        if len(self.data['reports']) > 0:
            if (self.data['survey_count'].iloc[0]['SurveyCount']) > 0:
                self.check_flag = True
                self.starting_date = self.update_window
                self.Logger.info(f"Getting emissions from {self.update_window} to {self.current_date}")

            else:
                self.Logger.info(f"No emissions found, processing from start")
                self.starting_date = STARTING_DATE
                self.check_flag = True
        else:
            self.Logger.info(f"No reports found, skipping")
            self.check_flag = False

    def query_data(self):
        if self.check_flag:
            # Ensure the ReportDate values are in datetime format before comparison,
            # handling both with and without microseconds (mixed formats)
            self.data['reports']['ReportDate'] = pd.to_datetime(self.data['reports']['ReportDate'], format='mixed')

            # Handle potential issues with type mismatch when comparing datetimes
            # Coerce both sides to date for a robust comparison
            reports_to_query = self.data['reports'][
                pd.to_datetime(self.data['reports']['ReportDate']).dt.date >= pd.to_datetime(self.starting_date).date()
            ]
   
            reports_to_query.db.set_query(query_surveys_table(report_table="#TempReports"))
            surveys = reports_to_query.db.execute(CONN_DICT[self.customer_info['DBLocation']], source_col = 'ReportId', temp_table_name = '#TempReports')
            surveys.db.set_query(query_segments_table(survey_table="#TempSurvey"))
            self.Logger.info(f"Surveys from LSDB: {len(surveys)}")
            segments = surveys.db.execute(CONN_DICT[self.customer_info['DBLocation']], source_col = 'SurveyId', temp_table_name = '#TempSurvey')
            self.Logger.info(f"Segments from LSDB: {len(segments)}")
            # Convert StartEpoch to datetime (time only, no date)
            segments['StartTime'] = pd.to_datetime(segments['StartEpoch'], unit='s').dt.time
            segments['StartDate'] = pd.to_datetime(segments['StartEpoch'], unit='s')
            segments['DayNight'] = segments['StartTime'].apply(lambda t: get_day_night(t, SUNRISE_TIME, SUNSET_TIME))
            segments['ActiveIdle'] = segments['CarSpeedMedian'].apply(lambda x: set_actie_idle(x, SPEED_THRESHOLD))

            # Set the starting time as a datetime object
            surveys['StartHour'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.hour
            surveys['StartTime'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.time
            surveys['EndHour'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.hour
            surveys['EndTime'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.time
            surveys['StartDay'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.date
            surveys['EndDay'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.date

            # Calculate the duration (in minutes) between StartEpoch and EndEpoch for each survey
            surveys['DurationMinutes'] = (
                surveys['EndEpoch'] - surveys['StartEpoch']
            ) / 60

            # Apply the function row-wise (axis=1) to build a DataFrame summary
            survey_summary = surveys.apply(survey_summary_apply, axis=1)
            segment_summary = segments.groupby("SurveyId").apply(segment_summary_apply)
            survey_summary = pd.merge(survey_summary,segment_summary,on="SurveyId",how="left")

            upload = survey_summary.reset_index(drop=True)
            upload.fillna(0, inplace=True)
            upload['LastUpdated'] = datetime.now()
            self.data['output'] = upload
        else:
            self.Logger.info(f"No reports found, skipping")
        
    def sanity_check(self):
        super().sanity_check()
        #Get all the reports from the KPI_SurveySummary table
        df_surveys = Query(query = f"SELECT DISTINCT ReportId FROM KPI_SurveySummary WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{self.customer_info['CustomerId']}')").execute(KPIHub_Conn)
        self.Logger.info(f"Total number of unique reports from KPI_SurveySummary: {len(df_surveys)}")

    def push_data(self):
        super().push_data(primary_key = ['SurveyId','ReportId'])
        self.Logger.info(f"Data pushed to {db_path}")

# Define a function to determine if survey is in 'day' or 'night'
def get_day_night(start_time, sunrise, sunset):
    # start_time should be a datetime.time object
    hour = start_time.hour
    if sunrise <= hour < sunset:
        return 'Day'
    else:
        return 'Night'

def set_actie_idle(speed, speed_threshold):
    if speed < speed_threshold:
        return 'Idle'
    else:
        return 'Active'

def segment_summary_apply(df):
    return pd.Series({
        'DaySegments': (df['DayNight'] == 'Day').sum(),
        'NightSegments': (df['DayNight'] == 'Night').sum(),
        'ActiveSegments': (df['ActiveIdle'] == 'Active').sum(),
        'IdleSegments': (df['ActiveIdle'] == 'Idle').sum(),
        'TotalSegments': len(df),
        'TotalKilometers': df['LengthMeters'].sum() / 1000,
        'DayKilometers': df.loc[df['DayNight'] == 'Day', 'LengthMeters'].sum() / 1000,
        'NightKilometers': df.loc[df['DayNight'] == 'Night', 'LengthMeters'].sum() / 1000,
        'SegmentDurationMinutes': df['DurationSeconds'].sum() / 60,
        'IdleTimeMinutes': df.loc[df['ActiveIdle'] == 'Idle', 'DurationSeconds'].sum() / 60,
        'ActiveTimeMinutes': df.loc[df['ActiveIdle'] == 'Active', 'DurationSeconds'].sum() / 60,
        'AvgSpeedKm': 3.6*df['CarSpeedMedian'].mean()

    })

def survey_summary_apply(row):
    return pd.Series({
        'SurveyId': row['SurveyId'],
        'SurveyorUnit': row['SurveyorUnit'],
        'SurveyDurationMinutes': row['DurationMinutes'],
        'ReportId': row['ReportId'],
        'StartHour': row['StartHour'],
        'StartTime': row['StartTime'],
        'StartEpoch': row['StartEpoch'],
        'EndTime': row['EndTime'],
        'EndEpoch': row['EndEpoch'],
        'StartDay': row['StartDay'],
        'EndDay': row['EndDay'],
        'LateralRotation': row['LateralRotation'],
        'NumberOfPeaks': row['NumberOfPeaks']
    })

In [67]:
customer_list = get_customer_list(KPIHub_Conn)
arguments = {'conn': KPIHub_Conn}
customer = customer_list.iloc[0]
surveyIngester = SurveySummaryIngester(arguments)
surveyIngester.set_customer_info(customer)
surveyIngester.update_check()
surveyIngester.query_data()
surveyIngester.push_data()
surveyIngester.sanity_check()

In [68]:
reports  = surveyIngester.data['reports']
reports.db.set_query(query_surveys_table(report_table="#TempReports"))
surveys = reports.db.execute(CONN_DICT[surveyIngester.customer_info['DBLocation']], source_col = 'ReportId', temp_table_name = '#TempReports') 
surveys.db.set_query(query_segments_table(survey_table="#TempSurvey"))
segments = surveys.db.execute(CONN_DICT[surveyIngester.customer_info['DBLocation']], source_col = 'SurveyId', temp_table_name = '#TempSurvey')

In [ ]:
from shapely import wkt
import geopandas as gpd
import pandas as pd

# One row per (ReportId, SurveyId) link
survey_report_links = surveys[['ReportId', 'SurveyId']].drop_duplicates()

# Load report areas once
report_ids = survey_report_links['ReportId'].unique().tolist()
report_areas = Query(
    query=f"""
    SELECT ReportId, ReportArea
    FROM KPI_ReportSummary
    WHERE ReportId IN ({','.join(repr(str(r)) for r in report_ids)})
    """
).execute(KPIHub_Conn)

report_areas = report_areas.dropna(subset=['ReportArea']).copy()
report_areas['geometry'] = report_areas['ReportArea'].apply(wkt.loads)
report_area_gdf = gpd.GeoDataFrame(report_areas[['ReportId', 'geometry']], geometry='geometry', crs='EPSG:4326')

# Segments with geometry; keep only surveys linked to a report
seg = segments[segments['SurveyId'].isin(survey_report_links['SurveyId'])].copy()
seg['geometry'] = seg['Shape'].apply(wkt.loads)
seg_gdf = gpd.GeoDataFrame(seg, geometry='geometry', crs='EPSG:4326')

# Project once for length-based tie-breaking
utm_crs = seg_gdf.estimate_utm_crs()
seg_utm = seg_gdf.to_crs(utm_crs)
report_utm = report_area_gdf.to_crs(utm_crs)

# Spatial join: segment may hit multiple report areas
joined = gpd.sjoin(
    seg_utm[['Id', 'SurveyId', 'geometry']],
    report_utm[['ReportId', 'geometry']],
    how='inner',
    predicate='intersects',
)

# Keep only report links that actually exist for that survey
joined = joined.merge(survey_report_links, on=['SurveyId', 'ReportId'], how='inner')

# Overlap length — assign each segment to the report with the largest intersection
report_geoms = report_utm.set_index('ReportId')['geometry']
joined = joined.copy()
joined['report_geom'] = joined['ReportId'].map(report_geoms)
joined['overlap_m'] = gpd.GeoSeries(joined.geometry, crs=utm_crs).intersection(
    gpd.GeoSeries(joined['report_geom'], crs=utm_crs)
).length

# One segment -> one report (no double count across overlapping report areas)
assigned = (
    joined.sort_values(['overlap_m', 'ReportId'], ascending=[False, True])
    .drop_duplicates(subset=['Id'], keep='first')
)

segments_in_area_df = (
    assigned.groupby(['ReportId', 'SurveyId'], as_index=False)
    .size()
    .rename(columns={'size': 'SegmentsInArea'})
)

# Include survey-report pairs with zero in-area segments
segments_in_area_df = survey_report_links.merge(
    segments_in_area_df, on=['ReportId', 'SurveyId'], how='left'
)
segments_in_area_df['SegmentsInArea'] = segments_in_area_df['SegmentsInArea'].fillna(0).astype(int)


In [ ]:
# Calculate segment counts per SurveyId
segments_per_survey = segments.groupby('SurveyId').size().reset_index(name='SegmentCount')
# Aggregate the SegmentsInArea sum per SurveyId
segments_in_area_by_survey = segments_in_area_df.groupby('SurveyId', as_index=False)['SegmentsInArea'].sum()

# Merge by SurveyId
merged = pd.merge(segments_per_survey, segments_in_area_by_survey, on='SurveyId', how='outer')

# Compare SegmentCount to SegmentsInArea
merged['CountMatches'] = merged['SegmentCount'] == merged['SegmentsInArea']

display(merged[~merged['CountMatches']])

,SurveyId,SegmentCount,SegmentsInArea,CountMatches
0,0040475F-B4E4-8EBB-44F9-3A210693FC9D,341,340,False
8,0C056150-5B76-07E9-A151-3A21BC1A369A,125,127,False
47,4105C020-3AD8-35BB-4293-3A2279F8A241,213,0,False
50,47A22A7F-2364-F7E4-FC60-3A211C5BD6FB,256,257,False
51,47E813AB-8C9B-A6BC-48B5-3A2101FF07A8,232,233,False
54,4A3B75AD-C4D6-55DF-86F6-3A21C0054BA6,120,122,False
62,502C7DAC-2CC1-976F-2C7B-3A20F7DD400B,306,307,False
65,57DE8D42-CB69-AE74-9A90-3A225077C8F1,382,384,False
73,61B378B1-1DDD-61E3-4B39-3A224B1EFF7F,374,375,False
74,647A21CE-A02D-7D87-BACE-3A2232323FFE,178,177,False
